# **Atelier Préparation de Données Images**

## **Partie 1 – Exploration du dataset**

### **1.1 Développer un programme Python capable de récupérer, pour chaque image, son nom, sa classe, son format, son mode, sa largeur, sa hauteur, l’écart-type de ses pixels, son nombre de canaux et sa taille.**

In [30]:
import os
import hashlib
import numpy as np
import pandas as pd
from PIL import Image, ImageOps

RAW_DIR = "../data/raw"

In [31]:
def get_image_info(filepath, classe):
    """Récupère les infos d'une image. Gère aussi le cas d'un fichier corrompu."""
    nom = os.path.basename(filepath)
    taille_octets = os.path.getsize(filepath)

    try:
        with Image.open(filepath) as img:
            img.load()  # force la lecture complète des pixels (détecte les fichiers corrompus)
            pixels = np.array(img)
            return {
                "nom": nom,
                "classe": classe,
                "format": img.format,
                "mode": img.mode,
                "largeur": img.width,
                "hauteur": img.height,
                "ecart_type_pixels": pixels.std(),
                "nb_canaux": len(img.getbands()),
                "taille_octets": taille_octets,
                "corrompue": False,
            }
    except Exception:
        return {
            "nom": nom,
            "classe": classe,
            "format": None,
            "mode": None,
            "largeur": None,
            "hauteur": None,
            "ecart_type_pixels": None,
            "nb_canaux": None,
            "taille_octets": taille_octets,
            "corrompue": True,
        }

In [32]:
lignes = []

for classe in sorted(os.listdir(RAW_DIR)):
    dossier_classe = os.path.join(RAW_DIR, classe)
    for nom_fichier in sorted(os.listdir(dossier_classe)):
        chemin_image = os.path.join(dossier_classe, nom_fichier)
        lignes.append(get_image_info(chemin_image, classe))

df_images = pd.DataFrame(lignes)

print(f"Nombre total d'images : {len(df_images)}")
df_images.head()

Nombre total d'images : 1026


,nom,classe,format,mode,largeur,hauteur,ecart_type_pixels,nb_canaux,taille_octets,corrompue
0,cardboard1.jpg,cardboard,JPEG,RGB,512.0,384.0,40.588529,3.0,17333,False
1,cardboard10.jpg,cardboard,JPEG,RGB,512.0,384.0,42.571288,3.0,21683,False
2,cardboard100.jpg,cardboard,JPEG,RGB,512.0,384.0,46.108305,3.0,14884,False
3,cardboard101.jpg,cardboard,JPEG,RGB,512.0,384.0,72.263996,3.0,14289,False
4,cardboard102.jpg,cardboard,JPEG,RGB,512.0,384.0,48.388937,3.0,18015,False


In [33]:
df_images[df_images['corrompue'] == True]

,nom,classe,format,mode,largeur,hauteur,ecart_type_pixels,nb_canaux,taille_octets,corrompue
147,cardboard83.jpg,cardboard,NaN,NaN,NaN,NaN,NaN,NaN,20888,True
323,glass74.jpg,glass,NaN,NaN,NaN,NaN,NaN,NaN,7184,True
441,metal48.jpg,metal,NaN,NaN,NaN,NaN,NaN,NaN,7259,True
627,paper213.jpg,paper,NaN,NaN,NaN,NaN,NaN,NaN,7293,True
785,plastic13.jpg,plastic,NaN,NaN,NaN,NaN,NaN,NaN,5250,True
998,trash3.jpg,trash,NaN,NaN,NaN,NaN,NaN,NaN,9192,True


## **Partie 2 – Détecter les images corrompues**

### **2.1 Ecriture d'une fonction qui détecte une image corrompue.**

In [34]:
def image_corrompu(file_path, classe):
    nom = os.path.basename(file_path)
    taille_octets = os.path.getsize(file_path)
    try:
        with Image.open(file_path) as img:
            img.load()  # exception levée si l'image est corrompue
            corrompu = False
    except Exception:
        corrompu = True

    return {
        "nom": nom,
        "taille": taille_octets,
        "classe": classe,
        "corrompu": corrompu,
    }

In [35]:
images_corrompu = []

for classe in sorted(os.listdir(RAW_DIR)):
    dossier_classe = os.path.join(RAW_DIR, classe)
    for nom_fichier in sorted(os.listdir(dossier_classe)):
        chemin_image = os.path.join(dossier_classe, nom_fichier)
        images_corrompu.append(image_corrompu(chemin_image, classe))

df_images_corrompu = pd.DataFrame(images_corrompu)
df_corrompues = df_images_corrompu[df_images_corrompu["corrompu"] == True]

print(f"Nombre total d'images : {len(df_images_corrompu)}")
print(f"Nombre d'images corrompues : {len(df_corrompues)}")
df_corrompues


Nombre total d'images : 1026
Nombre d'images corrompues : 6


,nom,taille,classe,corrompu
147,cardboard83.jpg,20888,cardboard,True
323,glass74.jpg,7184,glass,True
441,metal48.jpg,7259,metal,True
627,paper213.jpg,7293,paper,True
785,plastic13.jpg,5250,plastic,True
998,trash3.jpg,9192,trash,True


## **Partie 3 – Détection des images vides**

### **3.1 Écrire et se servir d’une fonction qui détecte les images vides : image entièrement noire, image entièrement blanche ou image dont les pixels présentent très peu de variation.**

In [36]:
print(df_images['taille_octets'].min(), df_images['taille_octets'].max())

924 221825


In [37]:
def image_vide(file_path, classe, seuil_std=5):
    """Détecte une image vide : entièrement noire, entièrement blanche, ou à très faible variation."""
    nom = os.path.basename(file_path)

    with Image.open(file_path) as img:
        img.load()
        pixels = np.array(img)

    entierement_noire = pixels.max() == 0
    entierement_blanche = pixels.min() == 255
    faible_variation = pixels.std() < seuil_std

    return {
        "nom": nom,
        "classe": classe,
        "ecart_type_pixels": pixels.std(),
        "vide": entierement_noire or entierement_blanche or faible_variation,
    }

In [38]:
images_valides = df_images[df_images["corrompue"] == False]

resultats_vide = []
for _, ligne in images_valides.iterrows():
    chemin_image = os.path.join(RAW_DIR, ligne["classe"], ligne["nom"])
    resultats_vide.append(image_vide(chemin_image, ligne["classe"]))

df_vide = pd.DataFrame(resultats_vide)
df_images_vides = df_vide[df_vide["vide"] == True]

print(f"Nombre d'images vides : {len(df_images_vides)}")
df_images_vides

Nombre d'images vides : 1


,nom,classe,ecart_type_pixels,vide
350,image-blanche-512x384.jpg,metal,1.572536,True


## **Partie 4 – Détecter les différences de résolution**

### **4.1 Déterminer la résolution minimale, la résolution maximale, les résolutions les plus fréquentes et le nombre d'images par résolution**

In [39]:
images_valides = df_images[df_images["corrompue"] == False].copy()
images_valides['resolution'] = list(zip(images_valides['largeur'], images_valides['hauteur']))
images_valides['nb_pixels'] = images_valides['hauteur'] * images_valides['largeur']

images_valides.head()

,nom,classe,format,mode,largeur,hauteur,ecart_type_pixels,nb_canaux,taille_octets,corrompue,resolution,nb_pixels
0,cardboard1.jpg,cardboard,JPEG,RGB,512.0,384.0,40.588529,3.0,17333,False,"(512.0, 384.0)",196608.0
1,cardboard10.jpg,cardboard,JPEG,RGB,512.0,384.0,42.571288,3.0,21683,False,"(512.0, 384.0)",196608.0
2,cardboard100.jpg,cardboard,JPEG,RGB,512.0,384.0,46.108305,3.0,14884,False,"(512.0, 384.0)",196608.0
3,cardboard101.jpg,cardboard,JPEG,RGB,512.0,384.0,72.263996,3.0,14289,False,"(512.0, 384.0)",196608.0
4,cardboard102.jpg,cardboard,JPEG,RGB,512.0,384.0,48.388937,3.0,18015,False,"(512.0, 384.0)",196608.0


#### **la resolution minimale & maximale**

In [40]:
resolution_min = images_valides.loc[images_valides["nb_pixels"].idxmin(), "resolution"]
resolution_max = images_valides.loc[images_valides["nb_pixels"].idxmax(), "resolution"]

print("Résolution minimale :", resolution_min)
print("Résolution maximale :", resolution_max)


Résolution minimale : (32.0, 32.0)
Résolution maximale : (512.0, 384.0)


#### **les Résolutions les plus fréquentes**

In [41]:
images_valides['resolution'].mode()

0    (512.0, 384.0)
Name: resolution, dtype: object

#### **Nombre d'image par resolution**

In [42]:
images_valides.groupby('resolution')['resolution'].value_counts()

resolution
(32.0, 32.0)         5
(40.0, 40.0)         4
(48.0, 32.0)         4
(512.0, 384.0)    1007
Name: count, dtype: int64

### **4.1 On décide qu'une image doit avoir au minimum 64 × 64 pixels. Identifier toutes les images ne respectant pas cette contrainte.**

In [43]:
print(f"On a {len(images_valides[images_valides['resolution'] < (64, 64)])} images qui ne respecte pas cette condition")
images_valides[images_valides['resolution'] < (64, 64)]

On a 13 images qui ne respecte pas cette condition


,nom,classe,format,mode,largeur,hauteur,ecart_type_pixels,nb_canaux,taille_octets,corrompue,resolution,nb_pixels
20,cardboard117.jpg,cardboard,JPEG,RGB,48.0,32.0,63.599233,3.0,1708,False,"(48.0, 32.0)",1536.0
77,cardboard22.jpg,cardboard,JPEG,RGB,32.0,32.0,52.236024,3.0,1247,False,"(32.0, 32.0)",1024.0
133,cardboard70.jpg,cardboard,JPEG,RGB,40.0,40.0,63.887915,3.0,1773,False,"(40.0, 40.0)",1600.0
169,glass100.jpg,glass,JPEG,RGB,40.0,40.0,70.101908,3.0,1591,False,"(40.0, 40.0)",1600.0
227,glass15.jpg,glass,JPEG,RGB,48.0,32.0,17.324620,3.0,924,False,"(48.0, 32.0)",1536.0
265,glass21.jpg,glass,JPEG,RGB,32.0,32.0,51.574317,3.0,1349,False,"(32.0, 32.0)",1024.0
267,glass23.jpg,glass,JPEG,RGB,32.0,32.0,54.756708,3.0,1254,False,"(32.0, 32.0)",1024.0
378,metal121.jpg,metal,JPEG,RGB,48.0,32.0,35.567019,3.0,1169,False,"(48.0, 32.0)",1536.0
408,metal2.jpg,metal,JPEG,RGB,32.0,32.0,49.339948,3.0,1116,False,"(32.0, 32.0)",1024.0
416,metal26.jpg,metal,JPEG,RGB,40.0,40.0,62.174044,3.0,1667,False,"(40.0, 40.0)",1600.0


## **Partie 5 – Détection des différents canaux**

### **5.1 On Détermine le nombre d'images selon leur nombre de canaux.**

In [44]:
images_valides.head(2)

,nom,classe,format,mode,largeur,hauteur,ecart_type_pixels,nb_canaux,taille_octets,corrompue,resolution,nb_pixels
0,cardboard1.jpg,cardboard,JPEG,RGB,512.0,384.0,40.588529,3.0,17333,False,"(512.0, 384.0)",196608.0
1,cardboard10.jpg,cardboard,JPEG,RGB,512.0,384.0,42.571288,3.0,21683,False,"(512.0, 384.0)",196608.0


In [45]:
image_par_canaux = images_valides['nb_canaux'].value_counts().reset_index()
image_par_canaux.columns = ['nb_canaux', 'nombre_images']

image_par_canaux

,nb_canaux,nombre_images
0,3.0,1003
1,4.0,17


## **Partie 6 – Détecter les doublons**

### **6.1 Écrire et se servir d’une fonction qui détecte les doublons**

In [46]:
import hashlib

In [47]:
def dectection_doublons(file_path, classe):
    """Calcule une empreinte (hash) à partir des pixels de l'image, pour repérer les doublons."""
    nom = os.path.basename(file_path)

    with Image.open(file_path) as img:
        img.load()
        pixels = np.array(img)

    hash_image = hashlib.md5(pixels.tobytes()).hexdigest()

    return {
        "nom": nom,
        "classe": classe,
        "hash": hash_image,
    }

In [48]:
resultats_hash = []
for _, ligne in images_valides.iterrows():
    chemin_image = os.path.join(RAW_DIR, ligne["classe"], ligne["nom"])
    resultats_hash.append(dectection_doublons(chemin_image, ligne["classe"]))

df_hash = pd.DataFrame(resultats_hash)
df_doublons = df_hash[df_hash.duplicated(subset="hash", keep=False)].sort_values("hash")

print(f"Nombre d'images en doublon : {len(df_doublons)}")
df_doublons

Nombre d'images en doublon : 20


,nom,classe,hash
969,plasticx199.jpg,plastic,020356755d7631716228a952bf3c3ead
858,plastic199.jpg,plastic,020356755d7631716228a952bf3c3ead
722,paper77ty.jpg,paper,116e03527a54192f3693d884d259d23b
721,paper77.jpg,paper,116e03527a54192f3693d884d259d23b
381,metal125po.jpg,metal,16e737d7e848620a6e96cc7e4f75dd37
380,metal125.jpg,metal,16e737d7e848620a6e96cc7e4f75dd37
676,paper35.jpg,paper,3596c22a22631f6cf70d5c039ce9cbc5
747,paperer35.jpg,paper,3596c22a22631f6cf70d5c039ce9cbc5
792,plastic140.jpg,plastic,3e56b719d0d3e0bb9decc95edfb21e8f
793,plastic140y.jpg,plastic,3e56b719d0d3e0bb9decc95edfb21e8f


## **Partie 7 – Détecter les images mal classées**

### **7.1 Effectuer un contrôle visuel pour détecter toute image mal classée.**

In [49]:
# Une image identique (même hash) présente dans deux classes différentes
# ne peut pas appartenir aux deux : au moins une des deux copies est mal classée.
# On s'appuie sur les hash calculés en Partie 6 pour cibler le contrôle visuel.
nb_classes_par_hash = df_hash.groupby("hash")["classe"].nunique().reset_index(name="nb_classes")
hash_suspects = nb_classes_par_hash[nb_classes_par_hash["nb_classes"] > 1]["hash"]

candidats_mal_classes = df_hash[df_hash["hash"].isin(hash_suspects)].sort_values("hash")

print(f"Nombre de candidats à vérifier visuellement : {len(candidats_mal_classes)}")
candidats_mal_classes

Nombre de candidats à vérifier visuellement : 0


,nom,classe,hash


**Contrôle visuel des candidats :**

- `image-violet-512x384.gif` (cardboard + glass), `image-blanche-512x384.jpg` (cardboard + metal), `image-noire-512x384.png` (glass + metal) : ce sont des images synthétiques de couleur unie, pas de vraies photos de déchets. Elles ne représentent aucune classe réelle → **mal classées dans les deux dossiers où elles apparaissent**.
- `glass115.jpg` / `metal91.jpg` : vue de dessus d'un goulot de bouteille avec un liseré sombre — visuellement plus proche d'une bouteille en **verre** → `metal91.jpg` semble mal classée.
- `glass176.jpg` / `plastic152.jpg` : bouteille striée semi-transparente avec étiquette — visuellement plus proche d'une bouteille en **plastique** (nervures typiques du PET) → `glass176.jpg` semble mal classée.


## **Partie 8 : Analyse du déséquilibre des classes**

### **On détermine le nombre d'images par classe**

In [50]:
print("Nombre d'image par classe")
images_valides['classe'].value_counts()

Nombre d'image par classe


classe
paper        251
plastic      223
glass        184
cardboard    166
metal        147
trash         49
Name: count, dtype: int64

## **Partie 9 : Redimensionnement**

### **9.1 Redimensionnement de toutes les images à 224 × 224: Redimensionnement avec conservation des proportions et ajout éventuel de padding**

In [51]:
from PIL import Image, ImageOps

In [52]:
CLEANED_DIR = "../data/cleaned"
TAILLE_CIBLE = (224, 224)

def redimensionner_et_sauvegarder(chemin_source, chemin_dest):
    """Redimensionne une image à 224x224 en gardant les proportions, avec du padding noir."""
    with Image.open(chemin_source) as img:
        img.load()
        img_redimensionnee = ImageOps.pad(img, TAILLE_CIBLE, color="black")
        img_redimensionnee = img_redimensionnee.convert("RGB")  # JPEG ne supporte pas RGBA
        os.makedirs(os.path.dirname(chemin_dest), exist_ok=True)
        img_redimensionnee.save(chemin_dest)

In [53]:
for _, ligne in images_valides.iterrows():
    chemin_source = os.path.join(RAW_DIR, ligne["classe"], ligne["nom"])
    chemin_dest = os.path.join(CLEANED_DIR, ligne["classe"], ligne["nom"])
    redimensionner_et_sauvegarder(chemin_source, chemin_dest)

print(f"{len(images_valides)} images redimensionnées et sauvegardées dans {CLEANED_DIR}")

1020 images redimensionnées et sauvegardées dans ../data/cleaned


## **Partie 10 : Uniformisation des canaux**

#### **Convertir toutes les images en RGB**

In [54]:
def convertir_en_rgb(file_path):
    """Convertit une image en RGB et la réenregistre."""
    with Image.open(file_path) as img:
        img_rgb = img.convert("RGB")
    img_rgb.save(file_path)


for classe in sorted(os.listdir(CLEANED_DIR)):
    dossier_classe = os.path.join(CLEANED_DIR, classe)
    for nom_fichier in sorted(os.listdir(dossier_classe)):
        convertir_en_rgb(os.path.join(dossier_classe, nom_fichier))

print("Conversion terminée : toutes les images de data/cleaned sont en RGB.")

Conversion terminée : toutes les images de data/cleaned sont en RGB.


In [55]:
# Vérification : tous les modes doivent être RGB
modes_cleaned = []
for classe in sorted(os.listdir(CLEANED_DIR)):
    dossier_classe = os.path.join(CLEANED_DIR, classe)
    for nom_fichier in sorted(os.listdir(dossier_classe)):
        with Image.open(os.path.join(dossier_classe, nom_fichier)) as img:
            modes_cleaned.append(img.mode)

pd.Series(modes_cleaned).value_counts()

RGB    1020
Name: count, dtype: int64

## **Partie 11 : Mise à l'échelle des pixels**

### **11.1 Normaliser les valeurs des pixels de toutes les images.**

In [56]:
# Les pixels vont de 0 à 255. On les ramène entre 0 et 1, ce qui aide un modèle à mieux apprendre.
# Note : le résultat est un tableau de floats, pas une image à sauvegarder sur le disque.
# Cette fonction sera utilisée au moment de charger les images pour l'entraînement.

def normaliser_image(file_path):
    """Charge une image et ramène ses pixels entre 0 et 1."""
    with Image.open(file_path) as img:
        pixels = np.array(img, dtype=np.float32)
    return pixels / 255.0

In [57]:
# Exemple sur une image pour vérifier que la normalisation fonctionne
dossier_exemple = os.path.join(CLEANED_DIR, "cardboard")
chemin_exemple = os.path.join(dossier_exemple, os.listdir(dossier_exemple)[0])

pixels_normalises = normaliser_image(chemin_exemple)

print("Forme du tableau :", pixels_normalises.shape)
print("Valeur min :", pixels_normalises.min())
print("Valeur max :", pixels_normalises.max())

Forme du tableau : (224, 224, 3)
Valeur min : 0.0
Valeur max : 1.0


## **Partie 12 : Découpage Train/Validation/Test**

### **12.1 Découper le dataset nettoyé (cleaned) en trois sous-ensembles : train, validation et test.**

In [58]:
from sklearn.model_selection import train_test_split

fichiers_cleaned = []
for classe in sorted(os.listdir(CLEANED_DIR)):
    dossier_classe = os.path.join(CLEANED_DIR, classe)
    for nom_fichier in sorted(os.listdir(dossier_classe)):
        fichiers_cleaned.append({"nom": nom_fichier, "classe": classe})

df_cleaned = pd.DataFrame(fichiers_cleaned)

In [59]:
# 70% train, 15% validation, 15% test — stratifié pour garder les mêmes proportions de classes partout
df_train, df_temp = train_test_split(
    df_cleaned, test_size=0.3, stratify=df_cleaned["classe"], random_state=42
)
df_val, df_test = train_test_split(
    df_temp, test_size=0.5, stratify=df_temp["classe"], random_state=42
)

print(f"Train : {len(df_train)} images")
print(f"Validation : {len(df_val)} images")
print(f"Test : {len(df_test)} images")

Train : 714 images
Validation : 153 images
Test : 153 images


In [60]:
# Vérification : les proportions de classes doivent rester similaires dans les 3 sous-ensembles
pd.DataFrame({
    "train": df_train["classe"].value_counts(normalize=True),
    "validation": df_val["classe"].value_counts(normalize=True),
    "test": df_test["classe"].value_counts(normalize=True),
}).round(3)

,train,validation,test
classe,,,
paper,0.246,0.248,0.242
plastic,0.218,0.216,0.222
glass,0.181,0.176,0.183
cardboard,0.162,0.163,0.163
metal,0.144,0.144,0.144
trash,0.048,0.052,0.046
